# Lưu grounded gain preparation — Output102 thành Dataset
**CPU / Internet ON / Save & Run All. Không GPU, không chạy lại chuẩn bị hay train.**
Tải đúng `ducanh2403/2-vi-cai-kia-qua-nhieu/102`, giữ nguyên5payload đã kiểm tra.
Không lấy latest khi thiếu file; không cần Add Input hoặc model checkpoint.
Cần khoảng5GiB trống cho bản tải và bản sao, gồm toàn bộ `inputs.h5`.
API token đọc từ Kaggle Secrets; không dán token vào code/chat.
Chạy bằng tài khoản có quyền xem Output102; OWNER phải khớp tài khoản sở hữu token.
SECRET_NAME là tên secret trong Add-ons → Secrets, khác tên gợi nhớ của token trên trang Settings.
Nếu secret của bạn tên `codex`, đổi SECRET_NAME thành `codex` và bật quyền cho notebook này.
Chỉ tạo dataset **Private mới**, không ghi đè dataset đã tồn tại.

In [ ]:
from pathlib import Path
KERNEL_VERSION='ducanh2403/2-vi-cai-kia-qua-nhieu/102'
OWNER='ducanh2403'
SLUG='grounded-gain-prep-v102'
TITLE='Grounded gain v102 feature dataset'
SECRET_NAME='KAGGLE_API_TOKEN'
DESTINATION=Path('/kaggle/working/grounded_gain_preparation_dataset')


In [ ]:
import os,re,subprocess,sys
from kaggle_secrets import UserSecretsClient
if KERNEL_VERSION!='ducanh2403/2-vi-cai-kia-qua-nhieu/102':raise ValueError('This notebook pins102; do not substitute latest or another version.')
if not re.fullmatch(r'[a-zA-Z0-9_-]+',OWNER) or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*',SLUG):raise ValueError('Invalid OWNER/SLUG')
if not 6<=len(SLUG)<=50 or not 6<=len(TITLE)<=50:raise ValueError('TITLE/SLUG must have6–50characters')
try:
    token=UserSecretsClient().get_secret(SECRET_NAME).strip()
    if not token:raise ValueError('empty')
except Exception:raise RuntimeError('Enable the configured SECRET_NAME in Add-ons > Secrets. Do not paste token in code/chat.') from None
def run_cli(args,authenticated=True):
    environment=os.environ.copy()
    for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(key,None)
    if authenticated:environment['KAGGLE_API_TOKEN']=token
    environment['PYTHONUNBUFFERED']='1'
    with subprocess.Popen(args,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace') as process:
        for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
        code=process.wait()
    if code:raise RuntimeError(f'Command failed ({code}); see redacted output. No automatic fallback or dataset overwrite.')

run_cli([sys.executable,'-m','pip','install','--upgrade','--force-reinstall','--quiet','https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip'],authenticated=False)
VERSION_PREFLIGHT="import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\n\n# Exercise the installed download implementation without contacting Kaggle.\napi = KaggleApi()\nwith tempfile.TemporaryDirectory() as directory, patch.object(api, 'build_kaggle_client') as factory:\n    endpoint = factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value = SimpleNamespace(files=[], next_page_token=None, log=None)\n    api.kernels_output('ducanh2403/2-vi-cai-kia-qua-nhieu/102', path=directory)\n    request = endpoint.call_args.args[0]\n    assert getattr(request, 'version_label', None) == 'v102', 'Installed CLI ignores the requested version; STOP before download.'\n    assert request.user_name == 'ducanh2403'\n    assert request.kernel_slug == '2-vi-cai-kia-qua-nhieu'\nprint('Version routing preflight PASS: request.version_label=v102; no network call.', flush=True)\n"
run_cli([sys.executable,'-u','-c',VERSION_PREFLIGHT])


In [ ]:
import hashlib,json
INVENTORY={'schema': 'grounded_gain_preparation_transport_v1', 'kernel_version': 'ducanh2403/2-vi-cai-kia-qua-nhieu/102', 'files': {'object_targets.json': 'e7a731b67dc04abe89a5a4d5a09a1da2b96300d503b89a5753e086100d747c92', 'inputs.h5': '582730b8dca6693294bbe6935d32d46b7f94a588eef3a6955809172db1a3794d', 'gain_metadata.json.gz': '62f43f771aec9feba31a2fbbb6fe7db2878f259979faac378b74a754822c7b67', 'initial_head.bin': 'e9bbacc34a70b785c36318f4e2149c423e6b0d972a680eba0e59aebc4a8d5169', 'receipt.json': 'd30c2ad5d4632f9e189c3607701defae62c48b21140f18d9c331febe09b1106e'}, 'source_review_zip_sha256': '3dedcbfd5ae930f56788ea40e521446dcbca2c7d4fd7476029e7fa461bf29e91'}
SOURCE_CODE='"""Verify and copy the completed preparation. Standard library, no API/model."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\n\nNAMES={\'receipt.json\',\'inputs.h5\',\'initial_head.bin\',\'gain_metadata.json.gz\',\'object_targets.json\'}\n\n\ndef sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda:f.read(8*1024**2),b\'\'):h.update(b)\n    return h.hexdigest()\n\n\ndef package(source,destination,inventory):\n    expected=inventory[\'files\']\n    if inventory[\'schema\']!=\'grounded_gain_preparation_transport_v1\' or set(expected)!=NAMES:\n        raise ValueError(\'Wrong inventory; refuse unknown or nested payload names\')\n    if any(not isinstance(h,str) or not re.fullmatch(\'[0-9a-f]{64}\',h) for h in expected.values()):raise ValueError(\'Invalid pinned hashes\')\n    source=Path(source).resolve();destination=Path(destination).resolve()\n    if source==destination or source in destination.parents or destination in source.parents:raise ValueError(\'Source and destination must be separate non-nested folders\')\n    for n,h in expected.items():\n        p=source/n\n        if p.is_symlink() or not p.is_file() or sha(p)!=h:raise ValueError(\'Missing/changed source: \'+n)\n        print(\'Verified source:\',n,flush=True)\n    receipt=json.loads((source/\'receipt.json\').read_text())\n    if receipt[\'status\']!=\'PREPARATION_COMPLETE\' or receipt[\'feature_extraction\'] or receipt[\'caption_training\']:raise ValueError(\'Wrong completed preparation\')\n    if receipt[\'payload_sha256\']!={n:h for n,h in expected.items() if n!=\'receipt.json\'}:raise ValueError(\'Payload manifest differs from pinned review\')\n    destination.mkdir(parents=True,exist_ok=True)\n    allowed=NAMES|{n+\'.partial\' for n in NAMES}|{\'transport_receipt.json\'}\n    if any(p.is_symlink() or p.name not in allowed for p in destination.iterdir()):raise ValueError(\'Unrelated destination files preserved; choose a fresh staging folder\')\n    needed=sum((source/n).stat().st_size for n in expected if not (destination/n).exists())\n    if shutil.disk_usage(destination).free<needed+256*1024**2:raise RuntimeError(\'Insufficient free space for verified copy\')\n    for n,h in expected.items():\n        target=destination/n\n        if target.exists():\n            if not target.is_file() or sha(target)!=h:raise ValueError(\'Changed destination preserved: \'+n)\n            continue\n        partial=destination/(n+\'.partial\');shutil.copyfile(source/n,partial)\n        if sha(partial)!=h:raise ValueError(\'Copied bytes differ: \'+n)\n        partial.replace(target);print(\'Copied and verified:\',n,flush=True)\n    if any(p.name not in NAMES|{\'transport_receipt.json\'} for p in destination.iterdir()):raise ValueError(\'Partial files remain; preserve and inspect staging\')\n    report=dict(status=\'DATASET_FOLDER_READY\',schema=inventory[\'schema\'],payload_files=5,\n        files=expected,bytes=sum((source/n).stat().st_size for n in expected),\n        source_kernel_version=inventory[\'kernel_version\'],source_review_zip_sha256=inventory[\'source_review_zip_sha256\'],\n        caption_training=False,feature_extraction=False,note=\'Five original payloads unchanged at dataset root; full HDF5 and initial-head bytes verified.\')\n    (destination/\'transport_receipt.json\').write_text(json.dumps(report,indent=2)+\'\\n\',encoding=\'utf-8\')\n    return report\n'
assert hashlib.sha256(SOURCE_CODE.encode()).hexdigest()=='faf70386b7662f5e6f511de226a70ccf628c58c559d9a214db823440b49f2c6f'
exec(compile(SOURCE_CODE,'package_gain_dataset.py','exec'))


In [ ]:
import tempfile,shutil
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<5*1024**3:raise RuntimeError('Need about5GiB free for download+copy; preserve existing output.')
DOWNLOAD=Path(tempfile.mkdtemp(prefix='grounded-gain-v102-',dir='/tmp'))
pattern=r'(^|/)(?:'+'|'.join(re.escape(n) for n in INVENTORY['files'])+r')$'
print('Downloading full preparation from:',KERNEL_VERSION,flush=True)
run_cli([sys.executable,'-m','kaggle','kernels','output',KERNEL_VERSION,'-p',str(DOWNLOAD),'--file-pattern',pattern])
matches=[p.parent for p in DOWNLOAD.rglob('receipt.json') if sha(p)==INVENTORY['files']['receipt.json']]
if len(matches)!=1:raise RuntimeError(f'Output102 returned {len(matches)} matching receipts; STOP, no latest/version fallback.')
report=package(matches[0],DESTINATION,INVENTORY)
print('All5original files verified. Ready for private dataset upload:',DESTINATION,flush=True)


In [ ]:
CREATE_DATASET_SCRIPT="import sys\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi = KaggleApi()\napi.authenticate()\nresult = api.dataset_create_new(sys.argv[1], public=False, quiet=False,\n                                convert_to_csv=False, dir_mode='skip')\nif result is None or str(getattr(result, 'status', '')).lower() != 'ok' or getattr(result, 'error', None):\n    raise RuntimeError('Dataset creation rejected: ' + str(getattr(result, 'error', 'Missing success response')))\nprint('Dataset creation accepted by API:', result.url, flush=True)\n"

metadata_path=DESTINATION/'dataset-metadata.json'
metadata=dict(title=TITLE,id=f'{OWNER}/{SLUG}',licenses=[dict(name='other')],
    description='Other license: private research preparation from pinned notebook Output102. Original COCO/model rights and upstream terms retained. Exact cached CLIP/original-prompt inputs for2000TRAIN+1000DEV, training-only object targets and common initial head. No trained checkpoint, new feature extraction or caption-quality claim. See original receipt and transport receipt.')
if metadata_path.exists():raise RuntimeError('Existing metadata preserved; choose fresh staging destination.')
try:
    metadata_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    run_cli([sys.executable,'-u','-c',CREATE_DATASET_SCRIPT,str(DESTINATION)])
finally:metadata_path.unlink(missing_ok=True)
print('API accepted creation; processing may still be pending. Do not repeatedly create.',flush=True)
print(f'https://www.kaggle.com/datasets/{OWNER}/{SLUG}',flush=True)
try:run_cli([sys.executable,'-m','kaggle','datasets','status',f'{OWNER}/{SLUG}'])
except RuntimeError:print('Status not confirmed; check dataset page before retrying.',flush=True)
try:run_cli([sys.executable,'-m','kaggle','datasets','files',f'{OWNER}/{SLUG}','--page-size','100'])
except RuntimeError:print('File list not available yet. Wait for processing; do not rerun create.',flush=True)


Sau khi Kaggle xử lý xong, kiểm tra5file gốc: `receipt.json`, `inputs.h5`,
`object_targets.json`, `gain_metadata.json.gz`, `initial_head.bin` (thêm transportreceipt).
Chia sẻ dataset Private cho Ngân; gắn cùng dataset vào C/H và T.
Cache nằm NGAY GỐC dataset, ví dụ:
`/kaggle/input/datasets/ducanh2403/grounded-gain-prep-v102`.
Không thêm `/grounded_gain_preparation` nếu file nằm ở gốc.
C/H đổi MODE='train', T giữ train; hai notebook T4x2 và cùng A/COCOJSON.
`PREPARED_ROOT=None` tự tìm receipt đúng; không phải chạy lại bước prepare.
Notebook này chưa được chạy API bởi Codex; chỉ kiểm tra local bằng file/secret giả lập.